In [1]:
import pandas as pd
import plotly.graph_objects as go

file_path = 'slide9.csv'  
df = pd.read_csv(file_path)

AGE_ORDER = [
    '≥70 anni',
    '65-69 anni',
    '60-64 anni',
    '55-59 anni',
    '50-54 anni',
    '45-49 anni',
    '40-44 anni',
    '35-39 anni',
    '30-34 anni',
    '25-29 anni',
    '20-24 anni',
    '15-19 anni',
]

LABEL_MAP = {
    '15-19 anni': '15-19 years',
    '20-24 anni': '20-24 years',
    '25-29 anni': '25-29 years',
    '30-34 anni': '30-34 years',
    '35-39 anni': '35-39 years',
    '40-44 anni': '40-44 years',
    '45-49 anni': '45-49 years',
    '50-54 anni': '50-54 years',
    '55-59 anni': '55-59 years',
    '60-64 anni': '60-64 years',
    '65-69 anni': '65-69 years',
    '≥70 anni':   '70+ years',
}

df = df[df['age_name'].isin(AGE_ORDER)].copy()
df['age_label'] = df['age_name'].map(LABEL_MAP)

years = sorted(df['year'].unique())

BAR_COLOR = "#bb87d0"   

def build_bar_trace(year):
    df_yr = (
        df[df['year'] == year]
        .set_index('age_name')
        .reindex(AGE_ORDER)
        .reset_index()
    )
    df_yr['age_label'] = df_yr['age_name'].map(LABEL_MAP)
    vals = df_yr['val'].fillna(0).round(2)
    labels = df_yr['age_label']

    return go.Bar(
        x=vals,
        y=labels,
        orientation='h',
        marker_color=BAR_COLOR,
        marker_line_width=0,
        text=[f'{v:.2f}%' for v in vals],
        textposition='outside',
        textfont=dict(size=12, color='#333333'),
        hovertemplate='<b>%{y}</b><br>%{x:.2f}%<extra></extra>',
        cliponaxis=False,
    )

fig = go.Figure()
fig.add_trace(build_bar_trace(years[0]))

frames = []
for yr in years:
    df_yr = (
        df[df['year'] == yr]
        .set_index('age_name')
        .reindex(AGE_ORDER)
        .reset_index()
    )
    df_yr['age_label'] = df_yr['age_name'].map(LABEL_MAP)
    vals = df_yr['val'].fillna(0).round(2)
    labels = df_yr['age_label']

    frame = go.Frame(
        name=str(yr),
        data=[go.Bar(
            x=vals,
            y=labels,
            orientation='h',
            marker_color=BAR_COLOR,
            marker_line_width=0,
            text=[f'{v:.2f}%' for v in vals],
            textposition='outside',
            textfont=dict(size=12, color='#333333'),
            hovertemplate='<b>%{y}</b><br>%{x:.2f}%<extra></extra>',
            cliponaxis=False,
        )],
        layout=go.Layout(
            xaxis=dict(range=[0, df['val'].max() * 1.18]),
        )
    )
    frames.append(frame)

fig.frames = frames

sliders = [dict(
    active=0,
    currentvalue=dict(
        prefix='',
        visible=True,
        xanchor='left',
        font=dict(size=13, color='#555555'),
    ),
    pad=dict(b=10, t=8),
    len=0.82,
    x=0.12,
    y=0,
    bgcolor='#eeeeee',
    activebgcolor='#555555',
    borderwidth=0,
    ticklen=4,
    tickcolor='#aaaaaa',
    steps=[
        dict(
            args=[[str(yr)],
                  dict(frame=dict(duration=350, redraw=True),
                       mode='immediate',
                       transition=dict(duration=150))],
            label=str(yr) if yr in [years[0], years[-1]] else '',
            method='animate',
        )
        for yr in years
    ],
)]

updatemenus = [dict(
    type='buttons',
    showactive=False,
    x=0.0,
    y=-0.03,
    xanchor='left',
    yanchor='top',
    buttons=[
        dict(
            label='▶',
            method='animate',
            args=[None, dict(
                frame=dict(duration=300, redraw=True), 
                mode='immediate',
                fromcurrent=True,
                transition=dict(duration=200),
            )],
        ),
        dict(
            label='⏸',
            method='animate',
            args=[[None], dict(
                frame=dict(duration=0, redraw=False),
                transition=dict(duration=0),
            )],
        ),
    ],
    font=dict(size=14),
    bgcolor='white',
    borderwidth=1,
    bordercolor='#cccccc',
)]

max_val = df['val'].max()

fig.update_layout(
    title=dict(
        text='Share of population with drug use disorders by age, 1990–2023',
        x=0.5,
        xanchor='center',
        font=dict(size=16, color='#222222'),
    ),
    xaxis=dict(
        range=[0, max_val * 1.18],
        showgrid=False,
        showticklabels=False,
        showline=False,
        zeroline=False,
        fixedrange=True,
    ),
    yaxis=dict(
        tickfont=dict(size=12, color='#333333'),
        showgrid=False,
        showline=False,
        zeroline=False,
        fixedrange=True,
        categoryorder='array',
        categoryarray=list(reversed([LABEL_MAP[a] for a in AGE_ORDER])),
    ),
    bargap=0.28,
    plot_bgcolor='white',
    paper_bgcolor='white',
    height=480,
    margin=dict(l=100, r=60, t=60, b=90),
    sliders=sliders,
    updatemenus=updatemenus,
)

fig.show()